# DEARLIBS Battery Model Implementation in Python

## Initialization

In [ ]:
# Imports

import numpy as np
import sympy as sp
from sympy import symbols, Function, diff, tanh, sinh, exp, sqrt, simplify
from sympy.utilities.lambdify import lambdify
from scipy.integrate import solve_ivp
from pyswarms.single import GlobalBestPSO
import matplotlib.pyplot as plt


### Model parameters, Applied current, and Number of node points.

In [ ]:
# Symbolic variables and parameters

t = sp.symbols('t')

# Number of parameters to be identified (Input the number of parameters to be identified)
n_vars = 8
kk = sp.symbols('kk1:%d' % (n_vars+1))  # k1, k2, ..., k8
# kk = sp.symbols('k1:9')  # k1 to k8

print(kk)

# C-rate (Put your C-rate)
Crate = -1

# Experimental data (Users can input their experimental conditions
Numexp = 63
Totexp = 3100

# Node number (Change # of node- N: Cathode, M: Membrance, NM: Cathode) (Put your number of node points)
N, M, NM = 2, 2, 2

# Design parameters
ep, es, en = 0.335, 0.47, 0.25
brugp, brugs, brugn = 2.43, 2.57, 2.91
lp, ls, ln1 = 75.6e-6, 12e-6, 85.2e-6
Rpp, Rpn = 5.22e-6, 5.86e-6
F = 96487
R = 8.3143
t1 = 0.363
ap = (3/Rpp)*(1-ep)
an = (3/Rpn)*(1-en)
T = 298.15
Acell = 0.11
Capa = 5
iapp = Capa * Crate / Acell

# Transport params symbolic with kk
c0 = 1000
D1 = kk[0] * 1e-9
Kappa = kk[1]
ctp = 51765
ctn = 29583
Dbulk = D1
sigmap = kk[2]
sigman = kk[3]
Dsp = kk[4] * 1e-15
Dsn = kk[5] * 1e-14

Keffp = Kappa * (ep ** brugp)
Keffs = Kappa * (es ** brugs)
Keffn = Kappa * (en ** brugn)
D2pos = (ep ** brugp) * Dbulk
D2sep = (es ** brugs) * Dbulk
D2neg = (en ** brugn) * Dbulk

kp = kk[6] * 1e-11
kn = kk[7] * 1e-12

h = lp/(N+1)
h2 = ls/(M+1)
h3 = ln1/(NM+1)

print(f"h  = {h}")
print(f"h2 = {h2}")
print(f"h3 = {h3}")


# Symbolic state variables X_i(t)
Nt = 1 + N + 1 + M + 1 + NM + 1 + N + NM + N + NM + N + 2 + NM + 2 + 1 + N + 1 + M + 1 + NM + 1

X = [Function(f'X{i+1}')(t) for i in range(Nt)]

## Define Equations

In [ ]:
# Create u1, u2, u3, u4, u5 arrays (mapping symbolic vars)


# Electrolyte concentration u1 (length = 1+N+1+M+1+NM+1)
u1_len = 1+N+1+M+1+NM+1
u1 = X[0:u1_len]

print('u1:', u1)

# Surface concentration u2 (length = N + NM)
u2 = [sp.sympify(0)] * 9

# First loop: i = 0 to 1 (N=2)
for i in range(N):
    index = i + (N + 1) + (M + 1) + (NM + 1)  # = i + 10
    u2[i + 1] = X[index + 1]  # u2[1] = X[11], u2[2] = X[12]

# Second loop: i = 0 to 1 (NM=2)
for i in range(NM):
    index = i + (N + 1) + (M + 1) + (NM + 1) + N  # = i + 12
    u2[i + 1 + N + 1 + M + 1] = X[index + 1]  # u2[7] = X[13], u2[8] = X[14]

print('u2:', u2)

# Average concentration u3 (length = N + NM)
u3 = [sp.sympify(0)] * 9
for i in range(N):
    u3[i + 1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + 1]

for i in range(1, NM+1):
    u3[i + 1 + N + 1 + M] = X[i + 1 + (N+1) + (NM+1) + N + NM + N + 2]

print('u3:', u3)

# Solid phase potential u4 (length = N + 2 + NM + 2)
u4 = [sp.sympify(0)] * 10
for i in range(N+2):
    u4[i] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM + 1]

for i in range(NM+2):
    u4[i + 1 + N + M + 1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM + N + N + 1]

print('u4:', u4)

# Liquid potential u5 (length = 1 + N + 1 + M + 1 + NM + 1)
u5 = [sp.sympify(0)] * 10
for i in range(10):
    u5[i] = X[i+1+N+1+M+1+NM+1+N+NM+N+NM+N+2+NM+2]
    
print('u5:', u5)    
    


In [ ]:
# Compute jp (molar flux)
# at positive electrode
jp = [sp.simplify(0)]*(N+1)
# jp = [sp.sympify(0)] * 10

for i in range(1, N+1):
    theta = u2[i]*ctp/ctp  # theta = u2(i)*ctp/ctp = u2(i)
    Up = (-0.8090)*theta + 4.4875 - 0.0428*tanh(18.5138*(theta-0.5542)) - 17.7326*tanh(15.7890*(theta-0.3117)) + 17.5842*tanh(15.9308*(theta-0.3120))
    jp[i] = 2*kp*sqrt(u1[i]*c0)*sqrt(ctp - u2[i]*ctp)*sqrt(u2[i]*ctp)*sinh(0.5*F/(R*T)*(u4[i] - u5[i] - Up))
    
# print('jp:', jp) 
print('len(jp):', len(jp))
    
# Display the results
print("jp = [")
for expr in jp:
    print(expr)
print("]")

In [ ]:
    
# at negative electrode
jn = [sp.simplify(0)]* (1 + N + 1 + M + 1 + NM)

# Loop indices
start_idx = 7
end_idx = 9

print(f"Length of jn: {len(jn)}")

for i in range(start_idx, end_idx):
    theta = u2[i]*ctn/ctn
    
    # Un = ((1.9793) * sp.exp(-39.3631 * theta) +
    #       0.2482 -
    #       0.0909 * sp.tanh(29.8538 * (theta - 0.1234)) -
    #       0.04478 * sp.tanh(14.9159 * (theta - 0.2769)) -
    #       0.0205 * sp.tanh(30.4444 * (theta - 0.6103)))
    
    Un = (0.2482
      + 1.9793 * theta * sp.exp(-39.3631 * theta)
      + 0.0909 * sp.tanh(29.8538 * (theta - 0.1234))
      - 0.04478 * sp.tanh(14.9159 * (theta - 0.2769))
      - 0.0205 * sp.tanh(30.4444 * (theta - 0.2769 - 0.6103)))
    
    jn[i] = 2*kn*sqrt(u1[i]*c0)*sqrt(ctn - u2[i]*ctn)*sqrt(u2[i]*ctn)*sinh(0.5*F/(R*T)*(u4[i] - u5[i] - Un))


# Display the results
print("jn = [")
for expr in jn:
    print(expr)
print("]")


In [ ]:
#  Form PDE/ODE equations (electrolyte concentration in positive electrode)
#u1: Electrolyte concentration (mol/m3)

# finite difference spatial derivatives approximations
dudxf1 = 1/(2*h) * (-u1[2] - 3*u1[0] + 4*u1[1])
dudxb1 = 1/(2*h) * (u1[N-1] + 3*u1[N+1] - 4*u1[N])
dudxf1_2 = 1/(2*h2) * (-u1[N+3] - 3*u1[N+1] + 4*u1[N+2])

bc11 = dudxf1
bc21 = D2pos * dudxb1 - D2sep * dudxf1_2


# Initialize eq1 list
eq1 = [sp.simplify(0)]* (1 + N + 1 + M + 1 + NM + 1)
print(eq1)
# eq1 = sp.zeros(1, 1 + N + 1 + M + 1 + NM + 1)

# eq1[0] = 0 - bc11
eq1[0] = sp.Eq(0, bc11)


for i in range(1, N):
    d2udx21 = (1 / h ** 2) * (u1[i-2] - 2 * u1[i-1] + u1[i])
    eq1[i-1] = sp.Eq(diff(u1[i-1]), (D2pos * d2udx21 + ap * (1 - t1) * jp[i-1] / c0) / ep)

# eq1[N + 1] = 0 - bc21

eq1[N + 1] = sp.Eq(0, bc21)

# Separator
dudxb1_2 = (1 / (2 * h2)) * (u1[N + M] + 3 * u1[N + M + 2] - 4 * u1[N + M + 1])
dudxf1_3 = (1 / (2 * h3)) * (-u1[N + M + 4] - 3 * u1[N + M + 2] + 4 * u1[N + M + 3])
bc31 = D2sep * dudxb1_2 - D2neg * dudxf1_3

for i in range(N + 2, N + M + 1):
    d2udx21 = (1 / h2 ** 2) * (u1[i - 2] - 2 * u1[i-1] + u1[i])
    eq1[i-1] = sp.Eq(diff(u1[i-1]), D2sep * d2udx21 / es)

# eq1[N + M + 2] = 0 - bc31
eq1[N + M + 2] = sp.Eq(0, bc31)

# Negative Electrode
dudxb1_3 = (1 / (2 * h3)) * (u1[1+N+1+M+1+NM-2] + 3 * u1[1+N+1+M+1+NM] - 4 * u1[1+N+1+M+1+NM-1])
bc41 = dudxb1_3

for i in range(N + M + 3, N + M + NM + 2):
    d2udx21 = (1 / h3 ** 2) * (u1[i - 2] - 2 * u1[i - 1] + u1[i])
    eq1[i-1] = sp.Eq(diff(u1[i-1]), (D2neg * d2udx21 + an * (1 - t1) * jn[i-1] / c0) / en)

# eq1[1+N+1+M+1+NM] = 0 - bc41
eq1[1+N+1+M+1+NM] = sp.Eq(0, bc41)

# u2: Surface concentration
eq2 = [sp.simplify(0)]* (N + M + NM + 3)

print(eq2)

# Positive electrode
for i in range(1, N):
    eq2[i-1] = sp.Eq(0,-u2[i-1] + u3[i-1] - jp[i-1] * Rpp / Dsp / 5 / ctp)

# Negative electrode
for i in range(N+2+M+1,N+2+M+1+NM-1):
    eq2[i-1] = sp.Eq(0,-u2[i-1] + u3[i-1] - jn[i-1] * Rpn / Dsn / 5 / ctn)

# u3: Average concentration
eq3 = [sp.simplify(0)]* (1+N+1+M+1+NM)
print(eq3)
for i in range(1, N):
    eq3[i-1] = sp.Eq(diff(u3[i-1]), - 3 * jp[i-1] / Rpp / ctp)

for i in range(1+N+1+M+1, 1+N+1+M+1+NM-1):
    eq3[i-1] = sp.Eq(diff(u3[i-1]), - 3 * jn[i-1] / Rpn / ctn)

# u4: Solid potential
eq4 = [sp.simplify(0)]* (1+N+1+M+1+NM+1)
print(eq4)

dudxf4 = (1 / (2 * h)) * (-u4[2] - 3 * u4[0] + 4 * u4[1])
dudxb4 = (1 / (2 * h)) * (u4[N - 1] + 3 * u4[N + 1] - 4 * u4[N])

bc14 = dudxf4 + iapp / sigmap
bc24 = dudxb4

# eq4[0] = 0 - bc14
eq4[0] = sp.Eq(0, bc14)

for i in range(1, N):
    d2udx24 = (1 / h ** 2) * (u4[i - 2] - 2 * u4[i-1] + u4[i])
    eq4[i-1] = sp.Eq(0,d2udx24 - ap * F * jp[i] / sigmap)

# eq4[N + 1] = 0 - bc24
eq4[N + 1] = sp.Eq(0, bc24)

# Negative
dudxf4_3 = (1 / (2 * h3)) * (-u4[N+2+M+1+1] - 3 * u4[N + M + 2] + 4 * u4[N + M + 3])
dudxb4_3 = (1 / (2 * h3)) * (u4[N + M + NM + 1] + 3 * u4[N + M + NM + 3] - 4 * u4[N + M + NM + 2])

bc34 = dudxf4_3
bc44 = dudxb4_3 + iapp / sigman

# eq4[N + M + 2] = 0 - bc34
eq4[N + M + 2] = sp.Eq(0, bc34)

for i in range(N+2+M+1, N + M + NM + 2):
    d2udx24 = (1 / h3 ** 2) * (u4[i - 2] - 2 * u4[i-1] + u4[i])
    eq4[i-1] = sp.Eq(0,d2udx24 - an * F * jn[i-1] / sigman)

# eq4[N + M + NM + 2] = 0 - bc44
eq4[1+N+1+M+1+NM] = sp.Eq(0, bc44)

# u5: Liquid phase potential (V)
# eq5 = sp.zeros(1, N + M + NM + 4)
eq5 = [sp.simplify(0)]* (1+N+1+M+1+NM+1)
print(eq5)

dudxf5 = (1 / (2 * h)) * (-u5[2] - 3 * u5[0] + 4 * u5[1])
dudxb5 = (1 / (2 * h)) * (u5[N - 1] + 3 * u5[N + 1] - 4 * u5[N])
dudxf5_2 = (1 / (2 * h2)) * (-u5[N + 3] - 3 * u5[N +1] + 4 * u5[N+2])

bc15 = dudxf5
bc25 = Keffp * dudxb5 - Keffs * dudxf5_2

# eq5[0] = 0 - bc15
eq5[0] = sp.Eq(0, bc15)

for i in range(1, N):
    dudx1 = (1 / (2 * h)) * (u1[i] - u1[i - 2])
    dudx4 = (1 / (2 * h)) * (u4[i] - u4[i - 2])
    dudx5 = (1 / (2 * h)) * (u5[i] - u5[i - 2])
    eq5[i-1] = sp.Eq(0,-sigmap * dudx4 - Keffp * dudx5 + (2 * Keffp * R * T * (1 - t1) * dudx1) / (F * u1[i-1]) - iapp)
    
eq5[N + 1] = sp.Eq(0, bc25)

# Separator
for i in range(N + 2, N + 1 + M):
    dudx1 = (1 / (2 * h2)) * (u1[i] - u1[i-2])
    dudx5 = (1 / (2 * h2)) * (u5[i] - u5[i-2])
    eq5[i - 1] = sp.Eq(0, -Keffs * dudx5 + (2 * Keffs * R * T * (1 - t1) * dudx1) / (F * u1[i-1]) - iapp)


dudxb5_2 = (1 / (2 * h2)) * (u5[N + M ] + 3 * u5[N + M + 2] - 4 * u5[N + M + 1])
dudxf5_3 = (1 / (2 * h3)) * (-u5[N + M + 4] - 3 * u5[N + M + 2] + 4 * u5[N + M + 3])
bc35 = Keffs * dudxb5_2 - Keffn * dudxf5_3

eq5[N + M + 2] = sp.Eq(0, bc35)

for i in range(N + M + 3, N + M + NM + 2):
    dudx1 = (1 / (2 * h3)) * (u1[i] - u1[i-2])
    dudx4 = (1 / (2 * h3)) * (u4[i] - u4[i-2])
    dudx5 = (1 / (2 * h3)) * (u5[i] - u5[i-2])
    eq5[i-1] = sp.Eq(0, -sigman * dudx4 - Keffn * dudx5 + (2 * Keffn * R * T * (1 - t1) * dudx1) / (F * u1[i-1]) - iapp)


bc45 = u5[N + M + NM + 3]
eq5[N + M + NM + 3] = sp.Eq(0, bc45)




## Optimization Setup

In [ ]:

# Python version of the MATLAB execution, solving, and optimization pipeline for the P2D model

import numpy as np
from sympy import Matrix, symbols, diag, lambdify
from scipy.integrate import solve_ivp
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt

# Placeholders for your symbolic variables (to be replaced by actual SymPy symbolic equations and variables)
eqn1, eqn2, eqn3, eqn4, eqn5 = [None]*5  # To be replaced with actual symbolic expressions
varsX = symbols('x0:100')  # Placeholder: adjust range and naming according to your model
kk = symbols('k0:8')

# Combine equations
eqs = eqn1 + eqn2[1:N+1] + eqn2[N+2+M+1:] + eqn3[1:N+1] + eqn3[N+2+M+1:] + eqn4[:N+2] + eqn4[1+N+1+M:] + eqn5

# Mass matrix formulation
MM_sym, f_sym = Matrix(eqs).as_explicit()

# Convert symbolic expressions to numerical functions
MM_func = lambdify((varsX, kk), MM_sym, modules='numpy')
f_func = lambdify((varsX, kk), f_sym, modules='numpy')

t = symbols('t')
mu = 1e-3
q = 1000
initime = 200
ff = 0.5 * np.tanh(q*(t - initime)) + 0.5

# Initial guess (adjust sizes according to actual model size)
U = np.zeros(1 + N + 1 + M + 1 + NM + 1 + 1 + N + 1 + M + 1 + NM + 1 + N + NM + N + NM + N + 2 + NM + 2)
U[:1+N+1+M+1+NM+1] = 1
U[1+1+N+1+M+1+NM+1:N+1+N+1+M+1+NM+1] = 0.27
U[1+1+N+1+M+1+NM+1+N:NM+1+N+1+M+1+NM+1+N] = 0.9014
U[1+1+N+1+M+1+NM+1+N+NM:N+1+N+1+M+1+NM+1+N+NM] = 0.27
U[1+1+N+1+M+1+NM+1+N+NM+N:NM+1+N+1+M+1+NM+1+N+NM+N] = 0.9014
U[1+1+N+1+M+1+NM+1+N+NM+N+NM:N+2+1+N+1+M+1+NM+1+N+NM+N+NM] = 4.30430037
U[1+1+N+1+M+1+NM+1+N+NM+N+NM+N+2:NM+2+1+N+1+M+1+NM+1+N+NM+N+NM+N+2] = 0.09202000152
U[1+1+N+1+M+1+NM+1+N+NM+N+NM+N+2+NM+2:] = 0

y0 = U.copy()

# Experimental data
x_exp = np.loadtxt('voltage_exp.txt')

# PSO bounds and parameters
pp = 0.3
init_params = [1, 1.17, 0.18, 215, 4, 3.3, 0.7, 0.7]
lower_bounds = [(1-pp)*p for p in init_params]
upper_bounds = [(1+pp)*p for p in init_params]

# Objective function
def P2Dobj(kk_):
    try:
        M0 = MM_func(y0, kk_)
        vw = 1 / np.maximum(np.abs(M0).max(axis=1), 1e-10)
        mw = np.diag(vw)

        def F(t, y):
            return vw * f_func(y, kk_)

        def M1(t, y):
            return mw @ MM_func(y, kk_)

        t_span = (0, Totexp + 200)
        t_eval = np.linspace(*t_span, Numexp + 3)

        sol = solve_ivp(F, t_span, y0, method='BDF', t_eval=t_eval, atol=1e-5, rtol=1e-5)
        V_model = sol.y[var1_index] - sol.y[var2_index]  # Fill with correct indices
        return np.sqrt(np.mean((x_exp[:, 0] - V_model[3:])**2))
    except:
        return 1000

# Run PSO (replaced with differential evolution for Python)
result = differential_evolution(P2Dobj, bounds=list(zip(lower_bounds, upper_bounds)), strategy='best1bin',
                                maxiter=10, popsize=10, disp=True)
kk_opt = result.x

# Extracted parameters
D1, Kappa, sigmap, sigman, Dsp, Dsn, kp, kn = kk_opt
D1 *= 1e-9
Dsp *= 1e-15
Dsn *= 1e-14
kp *= 1e-11
kn *= 1e-12



## Simulate the Model

In [ ]:
# Final solve using optimal params
M0 = MM_func(y0, kk_opt)
vw = 1 / np.maximum(np.abs(M0).max(axis=1), 1e-10)
mw = np.diag(vw)

F = lambda t, y: vw * f_func(y, kk_opt)
M1 = lambda t, y: mw @ MM_func(y, kk_opt)

sol = solve_ivp(F, (0, 100000), y0, method='BDF', atol=1e-5, rtol=1e-5)

## Plotting and Validation

# Plot
plt.figure(figsize=(10, 6))
plt.plot(sol.t - initime, sol.y[var1_index] - sol.y[var2_index], label='P2D Model', linewidth=2)
plt.plot(np.linspace(0, Totexp, Numexp), x_exp[:, 0], 'ro', label='Experiment')
plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')
plt.legend()
plt.grid(True)
plt.savefig('voltage_25C.bmp')
plt.show()